# PolygonConcavityIndex

[Codility task](https://app.codility.com/programmers/trainings/4/polygon_concavity_index/) - Exercise 4, Algorithmic skills

`A` is a list of 2-D points describing a simple polygon (consecutive points are joined, and the last one joins the first). Consecutive edges may be collinear.

`solution(A)` returns `-1` if the polygon is convex; otherwise it returns the index of **any** vertex that does not lie on the border of the convex hull of the points.

Example: the 7-point polygon `(-1,3) (1,2) (1,1) (3,1) (0,-1) (-2,1) (-1,2)` is concave and the answer is `2` or `6`.

Assumptions: `N` in `[3..10,000]`, coordinates up to `10^9`, no crossing edges, no duplicate points.

## Idea: convex hull with collinear points kept

A polygon is concave exactly when some vertex lies **strictly inside** the convex hull. So:

1. compute the hull border with **Andrew's monotone chain** over point **indices**;
2. pop only on a strictly right turn (`cross < 0`) so that points lying on a hull edge (180-degree angles) stay on the border;
3. return the first index that is not in the lower or upper chain, or `-1` if all vertices are on the border.

Time O(N log N) (the sort), space O(N).

In [ ]:
from dataclasses import dataclass


@dataclass
class Point2D:
    x: int
    y: int


def cross(o, a, b):
    """z-component of (a - o) x (b - o): > 0 left turn, < 0 right turn, 0 collinear."""
    return (a.x - o.x) * (b.y - o.y) - (a.y - o.y) * (b.x - o.x)


def solution(A):
    order = sorted(range(len(A)), key=lambda i: (A[i].x, A[i].y))

    def chain(indices):
        h = []
        for i in indices:
            while len(h) >= 2 and cross(A[h[-2]], A[h[-1]], A[i]) < 0:
                h.pop()
            h.append(i)
        return h

    border = set(chain(order)) | set(chain(reversed(order)))
    for i in range(len(A)):
        if i not in border:
            return i
    return -1

## Official examples

In [ ]:
def pts(*coords):
    return [Point2D(x, y) for x, y in coords]


convex = pts((-1, 3), (1, 2), (3, 1), (0, -1), (-2, 1))
assert solution(convex) == -1

concave = pts((-1, 3), (1, 2), (1, 1), (3, 1), (0, -1), (-2, 1), (-1, 2))
assert solution(concave) in (2, 6)
print("examples OK ->", solution(concave))

## Independent reference

To check the solution without using a convex hull, two geometric tests:

- **`strictly_inside(A, i)`**: a point is strictly inside the convex hull of the set iff the directions to all the other points are *not* contained in a closed half-plane, i.e. the largest angular gap between consecutive directions is **less than 180 degrees**. A gap of exactly 180 degrees means the point lies on the border (e.g. on an edge).
- **`convex_by_turns(A)`**: for a simple polygon, convexity means all turns have the same orientation (zeros allowed for 180-degree angles).

In [ ]:
import math


def strictly_inside(A, i):
    angles = sorted(
        math.atan2(q.y - A[i].y, q.x - A[i].x) for j, q in enumerate(A) if j != i
    )
    gaps = [b - a for a, b in zip(angles, angles[1:])] + [angles[0] + 2 * math.pi - angles[-1]]
    return max(gaps) < math.pi - 1e-9


def convex_by_turns(A):
    n = len(A)
    signs = {
        (c > 0) - (c < 0)
        for c in (cross(A[i], A[(i + 1) % n], A[(i + 2) % n]) for i in range(n))
    } - {0}
    return len(signs) <= 1


assert strictly_inside(concave, 2) and strictly_inside(concave, 6)
assert not any(strictly_inside(convex, i) for i in range(len(convex)))
assert convex_by_turns(convex) and not convex_by_turns(concave)
print("reference OK")

## Edge cases

In [ ]:
triangle = pts((0, 0), (4, 0), (0, 3))
assert solution(triangle) == -1

square = pts((0, 0), (4, 0), (4, 4), (0, 4))
assert solution(square) == -1

# collinear vertices on the border (180-degree angles) are still convex
square_with_midpoints = pts((0, 0), (2, 0), (4, 0), (4, 2), (4, 4), (2, 4), (0, 4), (0, 2))
assert solution(square_with_midpoints) == -1
assert solution(list(reversed(square_with_midpoints))) == -1          # clockwise order
assert solution(square_with_midpoints[3:] + square_with_midpoints[:3]) == -1   # another start vertex

# an "L" shape: the reflex vertex (1, 1) has index 3
l_shape = pts((0, 0), (4, 0), (4, 1), (1, 1), (1, 4), (0, 4))
assert solution(l_shape) == 3

# an "arrow head": the notch (2, 1) is strictly inside the hull
arrow = pts((0, 0), (2, 1), (4, 0), (2, 4))
assert solution(arrow) == 1

# extreme coordinates (cross products near 8 * 10**18)
big = pts((-10**9, -10**9), (10**9, -10**9), (10**9, 10**9), (-10**9, 10**9))
assert solution(big) == -1
big_concave = pts((-10**9, -10**9), (10**9, -10**9), (10**9, 10**9), (0, 0), (-10**9, 10**9))
assert solution(big_concave) == 3
print("edge cases OK")

## Random polygons against the reference

Simple polygons are generated in two ways:

1. **Star-shaped:** random distinct points sorted by angle around their centroid. With distinct angles this is always a simple polygon (usually concave).
2. **Convex:** the hull of random points, optionally with integer midpoints inserted on edges, in random orientation and with a random starting vertex.

For every polygon we check:

- if the polygon is convex (`convex_by_turns`), the solution returns `-1`, and no vertex is strictly inside;
- otherwise the solution returns an index that `strictly_inside` confirms.

In [ ]:
import random


def star_polygon(rng, n, span):
    cells = rng.sample([(x, y) for x in range(-span, span + 1) for y in range(-span, span + 1)], n)
    cx = sum(x for x, _ in cells) / n
    cy = sum(y for _, y in cells) / n
    cells.sort(key=lambda p: math.atan2(p[1] - cy, p[0] - cx))
    angles = [math.atan2(y - cy, x - cx) for x, y in cells]
    if any(b - a < 1e-9 for a, b in zip(angles, angles[1:])):       # tied angles: not a safe star
        return None
    polygon = pts(*cells)
    if all(cross(polygon[0], polygon[1], p) == 0 for p in polygon[2:]):   # all collinear: no polygon
        return None
    return polygon


def hull_by_wrapping(points):
    """Strict convex hull by gift wrapping (counter-clockwise), independent of the solution."""
    start = min(points)
    hull, current = [], start
    while True:
        hull.append(current)
        candidate = points[0] if points[0] != current else points[1]
        for p in points:
            if p == current:
                continue
            c = (candidate[0] - current[0]) * (p[1] - current[1]) - (candidate[1] - current[1]) * (p[0] - current[0])
            far = (p[0] - current[0]) ** 2 + (p[1] - current[1]) ** 2 > (candidate[0] - current[0]) ** 2 + (candidate[1] - current[1]) ** 2
            if c < 0 or (c == 0 and far):
                candidate = p
        current = candidate
        if current == start:
            return hull


def convex_polygon(rng, n, span):
    points = list({(rng.randint(-span, span), rng.randint(-span, span)) for _ in range(n)})
    if len(points) < 3:
        return None
    hull = hull_by_wrapping(points)
    if len(hull) < 3:
        return None
    with_mid = []
    for a, b in zip(hull, hull[1:] + hull[:1]):
        with_mid.append(a)
        dx, dy = b[0] - a[0], b[1] - a[1]
        if dx % 2 == 0 and dy % 2 == 0 and rng.random() < 0.7:
            with_mid.append((a[0] + dx // 2, a[1] + dy // 2))
    if rng.random() < 0.5:
        with_mid.reverse()
    k = rng.randrange(len(with_mid))
    return pts(*(with_mid[k:] + with_mid[:k]))


def check(A):
    inside = {i for i in range(len(A)) if strictly_inside(A, i)}
    is_convex = convex_by_turns(A)
    assert is_convex == (not inside), (A, inside)                  # the statement's claim
    result = solution(A)
    if is_convex:
        assert result == -1, A
    else:
        assert result in inside, (A, result, inside)
    return is_convex


rng = random.Random(10)
counts = {"convex": 0, "concave": 0}
for _ in range(3000):
    n = rng.randint(3, 12)
    A = star_polygon(rng, n, 6)
    if A is None:
        continue
    counts["convex" if check(A) else "concave"] += 1
for _ in range(1500):
    A = convex_polygon(rng, rng.randint(3, 20), 6)
    if A is not None:
        assert check(A) is True
        counts["convex"] += 1
print("random polygons verified:", counts)

## Large inputs

- a strictly convex polygon with 10,000 vertices on a parabola (`(i, i*i)`): the answer must be `-1`;
- a star-shaped polygon with 10,000 random vertices and coordinates up to `10^9`: the returned index is confirmed with the O(N) angular test.

In [ ]:
import time

parabola = pts(*[(i, i * i) for i in range(-5000, 5000)])
start = time.perf_counter()
assert solution(parabola) == -1
print(f"convex, N = {len(parabola)}: -1 in {time.perf_counter() - start:.3f}s")

rng = random.Random(21)
cells = set()
while len(cells) < 10_000:
    cells.add((rng.randint(-10**9, 10**9), rng.randint(-10**9, 10**9)))
cells = list(cells)
cx = sum(x for x, _ in cells) / len(cells)
cy = sum(y for _, y in cells) / len(cells)
cells.sort(key=lambda p: math.atan2(p[1] - cy, p[0] - cx))      # star-shaped around the centroid
big_star = pts(*cells)

start = time.perf_counter()
result = solution(big_star)
elapsed = time.perf_counter() - start
assert result != -1 and strictly_inside(big_star, result)
print(f"star-shaped, N = {len(big_star)}: index {result} (confirmed strictly inside) in {elapsed:.3f}s")

## Notes

- Concave means "some vertex is inside the hull". The statement guarantees such a vertex exists for every concave simple polygon, and the random test above checks this claim empirically.
- Pop on `cross < 0` (not `<= 0`) to keep collinear border points; popping on `<= 0` would give the strict hull and wrongly report 180-degree vertices as interior.
- Work with indices: the hull needs to return *which* vertex, not its coordinates.
- Integer arithmetic avoids the floating-point trouble of angle comparisons; the float `atan2` is used only in the test reference, with small coordinates.